# 📥 Download CAR - Google Colab

Este notebook permite baixar arquivos do Cadastro Ambiental Rural (SICAR) diretamente no Google Colab.

## 🎯 Objetivo
Permitir o download programático dos dados públicos do SICAR através de uma interface amigável no Google Colab.

## 📋 Pré-requisitos
- Conexão com internet
- Conta Google (para salvar arquivos no Drive, opcional)

---

## 🔧 Configuração Inicial

### 1. Montar Google Drive (Opcional)
Para salvar os arquivos permanentemente no seu Google Drive, execute a célula abaixo:

In [ ]:
# Montar Google Drive (opcional)
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive montado com sucesso!")

### 2. Instalar Dependências
Instalar o pacote download-car e suas dependências:

In [ ]:
# Instalar dependências do sistema
!sudo apt update
!sudo apt install -y tesseract-ocr tesseract-ocr-por

# Instalar o pacote download-car
!pip install git+https://github.com/Malnati/download-car

print("✅ Dependências instaladas com sucesso!")

### 3. Importar Bibliotecas

In [ ]:
import os
import zipfile
import shutil
from pathlib import Path
from datetime import datetime

# Importar classes do download-car
from download_car import DownloadCar, State, Polygon
from download_car.drivers import Tesseract, Paddle

print("✅ Bibliotecas importadas com sucesso!")

## ⚙️ Configuração dos Parâmetros

Configure os parâmetros de download abaixo:

In [ ]:
# ===== CONFIGURAÇÃO DOS PARÂMETROS =====

# Estado a ser baixado (escolha uma das opções abaixo)
ESTADO = "SP"  # Exemplos: "SP", "RJ", "MG", "PA", "DF", etc.

# Tipo de polígono (escolha uma das opções abaixo)
POLIGONO = "APPS"  # Exemplos: "APPS", "AREA_PROPERTY", "LEGAL_RESERVE", etc.

# Pasta de destino (opcional - deixe vazio para usar pasta padrão)
PASTA_DESTINO = ""  # Exemplo: "drive/MyDrive/download-car/SP"

# Configurações avançadas
TENTATIVAS = 25      # Número máximo de tentativas
TIMEOUT = 30         # Timeout em segundos
DEBUG = False        # Modo debug (True/False)
DRIVER = "Tesseract" # Driver OCR: "Tesseract" ou "Paddle"

# ======================================

print("📋 Parâmetros configurados:")
print(f"   Estado: {ESTADO}")
print(f"   Polígono: {POLIGONO}")
print(f"   Pasta: {PASTA_DESTINO if PASTA_DESTINO else 'Padrão'}")
print(f"   Tentativas: {TENTATIVAS}")
print(f"   Timeout: {TIMEOUT}s")
print(f"   Debug: {DEBUG}")
print(f"   Driver: {DRIVER}")

## ✅ Validação dos Parâmetros

Verificar se os parâmetros são válidos:

In [ ]:
# Validar estado
try:
    state_enum = State[ESTADO]
    print(f"✅ Estado válido: {ESTADO} ({state_enum.value})")
except KeyError:
    print(f"❌ Estado inválido: {ESTADO}")
    print("Estados disponíveis:")
    for state in State:
        print(f"  - {state.name}: {state.value}")
    raise ValueError(f"Estado '{ESTADO}' não encontrado")

# Validar polígono
try:
    polygon_enum = Polygon[POLIGONO]
    print(f"✅ Polígono válido: {POLIGONO} ({polygon_enum.value})")
except KeyError:
    print(f"❌ Polígono inválido: {POLIGONO}")
    print("Polígonos disponíveis:")
    for polygon in Polygon:
        print(f"  - {polygon.name}: {polygon.value}")
    raise ValueError(f"Polígono '{POLIGONO}' não encontrado")

# Validar driver
if DRIVER not in ["Tesseract", "Paddle"]:
    print(f"❌ Driver inválido: {DRIVER}")
    raise ValueError("Driver deve ser 'Tesseract' ou 'Paddle'")
else:
    print(f"✅ Driver válido: {DRIVER}")

print("\n🎯 Todos os parâmetros são válidos!")
print("Pronto para iniciar o download.")

## 🚀 Executar Download

Iniciar o processo de download:

In [ ]:
# Configurar pasta de destino
if PASTA_DESTINO:
    folder = PASTA_DESTINO
else:
    folder = f"temp/{ESTADO}"

# Criar pasta se não existir
os.makedirs(folder, exist_ok=True)

# Selecionar driver
driver = Tesseract if DRIVER == "Tesseract" else Paddle

print(f"🚀 Iniciando download...")
print(f"   Estado: {ESTADO}")
print(f"   Polígono: {POLIGONO}")
print(f"   Pasta: {folder}")
print(f"   Driver: {DRIVER}")
print("\n⏳ Aguarde, o download pode levar alguns minutos...")

# Criar instância do DownloadCar
car = DownloadCar(driver=driver)

# Executar download
try:
    result = car.download_state(
        state=state_enum,
        polygon=polygon_enum,
        folder=folder,
        tries=TENTATIVAS,
        debug=DEBUG,
        timeout=TIMEOUT
    )
    
    print("\n✅ Download concluído com sucesso!")
    print(f"📁 Arquivos salvos em: {folder}")
    
except Exception as e:
    print(f"\n❌ Erro durante o download: {str(e)}")
    raise

## 📁 Verificar Arquivos Baixados

Listar os arquivos que foram baixados:

In [ ]:
# Verificar arquivos na pasta de destino
print(f"📁 Conteúdo da pasta: {folder}")
print("=" * 50)

if os.path.exists(folder):
    files = os.listdir(folder)
    if files:
        for file in sorted(files):
            file_path = os.path.join(folder, file)
            size = os.path.getsize(file_path)
            size_mb = size / (1024 * 1024)
            print(f"📄 {file} ({size_mb:.2f} MB)")
    else:
        print("❌ Nenhum arquivo encontrado na pasta")
else:
    print(f"❌ Pasta não encontrada: {folder}")

## 📦 Criar Arquivo ZIP

Criar um arquivo ZIP com todos os arquivos baixados para facilitar o download:

In [ ]:
# Criar nome do arquivo ZIP
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_filename = f"{ESTADO}_{POLIGONO}_{timestamp}.zip"
zip_path = f"/content/{zip_filename}"

print(f"📦 Criando arquivo ZIP: {zip_filename}")

# Criar arquivo ZIP
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    if os.path.exists(folder):
        for root, dirs, files in os.walk(folder):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, folder)
                zipf.write(file_path, arcname)
                print(f"  📄 Adicionado: {arcname}")

# Verificar tamanho do ZIP
zip_size = os.path.getsize(zip_path) / (1024 * 1024)
print(f"\n✅ Arquivo ZIP criado: {zip_filename} ({zip_size:.2f} MB)")

## 💾 Download do Arquivo ZIP

Baixar o arquivo ZIP para o seu computador:

In [ ]:
from google.colab import files

print(f"💾 Iniciando download do arquivo: {zip_filename}")
print(f"📊 Tamanho: {zip_size:.2f} MB")
print("\n⏳ O download será iniciado automaticamente...")

# Fazer download do arquivo
files.download(zip_path)

print("\n✅ Download iniciado! Verifique a pasta de downloads do seu navegador.")

## 📅 Informações Adicionais

### Datas de Lançamento
Verificar as datas de lançamento dos dados:

In [ ]:
# Obter datas de lançamento
print("📅 Datas de lançamento dos dados:")
print("=" * 50)

release_dates = car.get_release_dates()

# Mostrar data do estado baixado
if state_enum in release_dates:
    print(f"🎯 {ESTADO}: {release_dates[state_enum]}")
else:
    print(f"❌ Data não disponível para {ESTADO}")

print("\n📋 Datas de todos os estados:")
for state, date in sorted(release_dates.items()):
    print(f"  {state.name}: {date}")

## 📚 Ajuda e Referências

### Estados Disponíveis
```python
from download_car import State
for state in State:
    print(f"{state.name}: {state.value}")
```

### Polígonos Disponíveis
```python
from download_car import Polygon
for polygon in Polygon:
    print(f"{polygon.name}: {polygon.value}")
```

### Links Úteis
- [SICAR - Sistema Nacional de Cadastro Ambiental Rural](https://www.car.gov.br/)
- [Repositório do Projeto](https://github.com/Malnati/download-car)
- [Documentação da API](https://github.com/Malnati/download-car-api)

---

## 🎉 Conclusão

O download foi concluído com sucesso! Os arquivos estão disponíveis:
1. **Na pasta local**: `{folder}`
2. **Como arquivo ZIP**: `{zip_filename}` (baixado automaticamente)

### Próximos Passos
- Abra o arquivo ZIP baixado
- Use os shapefiles em seu software GIS preferido (QGIS, ArcGIS, etc.)
- Para baixar outros estados ou polígonos, altere os parâmetros na seção "Configuração" e execute novamente

**Dúvidas?** Consulte a documentação do projeto ou abra uma issue no GitHub.